# Synapse Google Colab GPU 实验工作流 (E1–E4)

本 Notebook 用于在 Google Colab (T4 / L4 / A100) 环境下一站式执行 **Synapse Layer 2** 3D CG 渲染与神经视频生成互操作实验：
- **E1**：FGPP 7 通道空间条件控制与消融（RGB / Depth / Normal / Motion / Occlusion）
- **E2**：ACEScg 内蕴风格提取与 16px 手术刀无缝合成
- **E3**：97 帧长时序因果滑窗、Sigmoid 潜变量融合与切镜重锚定
- **E4**：WebDataset 潜变量特征分片与 Cryptomatte Multi-LoRA 训练

> **持久化保障**：所有实验状态与 LoRA 权重直接写回 Google Drive，会话掉线可无缝断点续跑。

In [ ]:
# 1. 挂载 Google Drive 持久化目录 (遵循 SSOT V2.0: 根目录零污染，统一 /dervgen/)
from google.colab import drive, userdata
import os
from pathlib import Path

drive.mount('/content/drive')

DERVGEN_DRIVE_ROOT = Path(os.environ.get('DERVGEN_DRIVE_ROOT', '/content/drive/MyDrive/dervgen'))
USER_ID = 'yanglei'  # 可根据实际使用者或 Agent 模型标识设置，如 gemini3.8, opus4.6
TODAY = '2026-11-12'

work_dir = DERVGEN_DRIVE_ROOT / 'work' / USER_ID / 'synapse'
work_dir.mkdir(parents=True, exist_ok=True)

# 配置安全凭据 (从 Colab Secrets 读取，严禁代码硬编码)
try:
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token:
        os.environ['GITHUB_TOKEN'] = gh_token
        print('✅ GITHUB_TOKEN loaded securely from Colab Secrets.')
except Exception:
    print('ℹ️ GITHUB_TOKEN not configured in Colab Secrets.')

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print('✅ HF_TOKEN loaded securely from Colab Secrets.')
except Exception:
    print('ℹ️ HF_TOKEN not configured in Colab Secrets.')

print(f'✅ Google Drive SSOT V2.0 workspace ready: {work_dir}')


In [ ]:
# 2. 获取代码并执行环境初始化脚本
# 如果仓库尚未存在，请取消以下注释进行克隆；或者从本地上传代码压缩包
# !git clone https://github.com/sautille-stroke/synapse.git /content/synapse
%cd /content/synapse

!chmod +x workenv/google-colab/setup_colab.sh
!bash workenv/google-colab/setup_colab.sh

In [ ]:
# 3. 硬件自检与模型选型档案自动侦测
!python scripts/colab_runner.py --check-env

In [ ]:
# 4. 下载或挂载 Golden Dataset 数据包 (优先从 Google Drive SSOT V2.0 路径读取)
import os, shutil
from pathlib import Path

    drive_bundle = DERVGEN_DRIVE_ROOT / 'data' / 'video' / 'synapse_colab_bundle_v1.tar.gz'
    if not drive_bundle.exists():
        drive_bundle = DERVGEN_DRIVE_ROOT / 'data' / 'benchmarks' / 'synapse' / 'synapse_colab_bundle_v1.tar.gz'
local_bundle = Path('/content/synapse_colab_bundle_v1.tar.gz')

if drive_bundle.is_file() and not local_bundle.is_file():
    print(f'Copying bundle from Google Drive: {drive_bundle} -> {local_bundle}')
    shutil.copy(drive_bundle, local_bundle)

!ls -lh /content/synapse_colab_bundle_v1.tar.gz 2>/dev/null || echo 'Bundle will be extracted if present'


In [ ]:
# 5. 执行全套实验套件 (严格输出至 SSOT V2.0 作业空间，带断点续跑)
from datetime import datetime
today_str = datetime.now().strftime('%Y-%m-%d')
job_id = 'exp01-wan-lora'
job_dir = DERVGEN_DRIVE_ROOT / 'work' / USER_ID / 'synapse' / today_str / job_id

!python scripts/colab_runner.py \
    --bundle /content/synapse_colab_bundle_v1.tar.gz \
    --dataset-dir /content/datasets/synapse \
    --job-dir {job_dir} \
    --user {USER_ID} \
    --job-id {job_id} \
    --exp all


In [ ]:
# 6. 审查与打印 Google Drive 中的任务元数据与收敛指标 (README.md & job.json)
import json
from pathlib import Path

job_json = job_dir / 'job.json'
if job_json.is_file():
    data = json.loads(job_json.read_text(encoding='utf-8'))
    print('=== job.json ===')
    print(json.dumps(data, indent=2, ensure_ascii=False))

readme_file = job_dir / 'README.md'
if readme_file.is_file():
    print('\n=== README.md Scorecard ===')
    print(readme_file.read_text(encoding='utf-8'))
